# Lab 2: Data Collection with Web Scraping

**Goal:** Scrape the Falcon 9 launch history table from Wikipedia and parse it into a clean DataFrame. This gives us a second, independent source of launch records (date, booster version, launch site, payload, orbit, outcome) to cross-check and enrich the API dataset from Lab 1.

We scrape a **fixed historical revision** of the Wikipedia page (not the live page) so the data doesn't shift under us while you're working.

In [1]:
!pip install beautifulsoup4 requests -q

In [2]:
import requests
import pandas as pd
import re
import unicodedata
from bs4 import BeautifulSoup

## 1. Helper functions to parse each table cell
Wikipedia's launch table packs a lot into each cell (dates with footnotes, booster version with links, etc). These helpers pull out just the clean value.

In [3]:
def date_time(table_cells):
    return [date_time.strip() for date_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg")+2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not column_name.strip().isdigit():
        column_name = column_name.strip()
        return column_name

## 2. Request the fixed Wikipedia revision and build the soup object

In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

response = requests.get(static_url, headers=headers)
print("Status code:", response.status_code)
print("Content length:", len(response.text))

soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title)

Status code: 200
Content length: 3282496
<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


## 3. Find all launch tables and extract column names

In [5]:
html_tables = soup.find_all('table')
print("Number of tables found:", len(html_tables))

if len(html_tables) == 0:
    print("No tables found — check status code and content length above.")
    print("First 500 chars of response:")
    print(response.text[:500])
else:
    # The launch records table is (usually) the 3rd table on the page (index 2).
    # If parsing looks wrong later, print a few tables' first rows to find the right index.
    first_launch_table = html_tables[2] if len(html_tables) > 2 else html_tables[0]

    column_names = []
    for th in first_launch_table.find_all('th'):
        name = extract_column_from_header(th)
        if name is not None and len(name) > 0:
            column_names.append(name)

    print(column_names)

Number of tables found: 25
['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## 4. Parse every row of every launch table into a dictionary

In [6]:
launch_dict = dict.fromkeys(column_names)

# Remove irrelevant columns and set up the fields we actually need
del launch_dict['Date and time ( )']

launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

In [7]:
extracted_row = 0

for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False

        row = rows.find_all('td')

        if flag:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)

            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1] if len(datatimelist) > 1 else None)

            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string if row[1].a else None
            launch_dict['Version Booster'].append(bv)

            launch_site = row[2].a.string if row[2].a else row[2].text.strip()
            launch_dict['Launch site'].append(launch_site)

            payload = row[3].a.string if row[3].a else row[3].text.strip()
            launch_dict['Payload'].append(payload)

            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)

            orbit = row[5].a.string if row[5].a else row[5].text.strip()
            launch_dict['Orbit'].append(orbit)

            customer = row[6].a.string if row[6].a else row[6].text.strip()
            launch_dict['Customer'].append(customer)

            launch_outcome = list(row[7].strings)[0].strip()
            launch_dict['Launch outcome'].append(launch_outcome)

            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print("Rows extracted:", extracted_row)

Rows extracted: 121


## 5. Build the DataFrame and save to CSV

In [8]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
df.head()

,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10


In [9]:
df.to_csv('spacex_web_scraped.csv', index=False)
print('Saved spacex_web_scraped.csv with', df.shape[0], 'rows')

Saved spacex_web_scraped.csv with 121 rows
